# Oracle ranking with recording first N seconds of performance metrics

In [7]:
import csv
import math
import re
from pathlib import Path
from collections import defaultdict
from contextlib import redirect_stdout
from io import StringIO

# ===== Paths (H100 only) =====
SPEC_ROOT_DIR = Path("/home/ac.zzheng/power/GPGPU/data/H100/spec_power_motif")
ML_ROOT_DIR   = Path("/home/ac.zzheng/power/GPGPU/data/H100/ml_power_motif")
TXT_OUT_PATH  = Path("/home/ac.zzheng/power/GPGPU/data/H100/h100_rank_results.txt")

SPEC_APPS = ["cloverleaf", "lbm", "minisweep", "miniweather", "pot3d", "tealeaf"]
ML_APPS = None  # None -> auto-discover

PROFILE_SECONDS = 15.0
ACTIVE_POWER_TH = 120.0
ACTIVE_SM_TH = 400.0

FILE_RE = re.compile(r"(?P<cap>\d+)_(?P<gpus>\d+)_gpu_metrics\.csv$")


def f(x):
    try:
        return float(str(x).strip())
    except Exception:
        return None


def mean(xs):
    return sum(xs) / len(xs) if xs else float("nan")


def fmt2(x):
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return "nan"
    return f"{x:.2f}"


def list_apps(root, apps):
    if apps is not None:
        return apps
    if not root.exists():
        return []
    return sorted([p.name for p in root.iterdir() if p.is_dir()])


def detect_metric_keys(app_dir):
    keys = set()
    for p in app_dir.iterdir():
        m = FILE_RE.match(p.name)
        if m:
            keys.add((int(m.group("cap")), int(m.group("gpus"))))
    return keys


def read_runtime_lookup_spec(app_dir):
    """
    Build lookup {(total_cap, gpu_count): runtime_seconds}
    Auto-detect whether runtime.csv power_cap is total-cap or per-gpu-cap by overlap with metric keys.
    """
    out = {}
    p = app_dir / "runtime.csv"
    if not p.exists():
        return out

    metric_keys = detect_metric_keys(app_dir)
    rows = []

    with p.open(newline="") as fh:
        r = csv.DictReader(fh)
        cols = set(r.fieldnames or [])
        req = {"power_cap", "gpu_count", "runtime_seconds"}
        if not req.issubset(cols):
            return out
        for row in r:
            cap = f(row.get("power_cap"))
            g = f(row.get("gpu_count"))
            v = f(row.get("runtime_seconds"))
            if cap is None or g is None or v is None:
                continue
            g = int(round(g))
            rows.append((cap, g, v))

    # candidate A: cap already total
    raw = {(int(round(cap)), g): v for cap, g, v in rows}
    # candidate B: cap is per-gpu, convert to total
    per = {(int(round(cap * g)), g): v for cap, g, v in rows}

    raw_hits = sum(1 for k in raw if k in metric_keys)
    per_hits = sum(1 for k in per if k in metric_keys)

    return raw if raw_hits >= per_hits else per


def read_throughput_lookup_ml(app_dir):
    """
    Build lookup {(total_cap, gpu_count): throughput}
    Supports throughput.csv columns:
      - total_gpu_cap (preferred)
      - power_cap (fallback)
      - throughput_images_per_sec / throughput_tokens_per_sec / throughput
    """
    out = {}
    p = app_dir / "throughput.csv"
    if not p.exists():
        return out

    with p.open(newline="") as fh:
        r = csv.DictReader(fh)
        cols = set(r.fieldnames or [])

        cap_col = None
        for c in ("total_gpu_cap", "power_cap"):
            if c in cols:
                cap_col = c
                break
        if cap_col is None or "gpu_count" not in cols:
            return out

        perf_col = None
        for c in ("throughput_images_per_sec", "throughput_tokens_per_sec", "throughput"):
            if c in cols:
                perf_col = c
                break
        if perf_col is None:
            return out

        has_status = "status" in cols
        for row in r:
            if has_status and str(row.get("status", "")).strip().lower() != "ok":
                continue
            cap = f(row.get(cap_col))
            g = f(row.get("gpu_count"))
            v = f(row.get(perf_col))
            if cap is None or g is None or v is None:
                continue
            out[(int(round(cap)), int(round(g)))] = v

    return out


def extract_metrics(metric_csv):
    m = FILE_RE.match(metric_csv.name)
    if not m:
        return None
    cap = int(m.group("cap"))
    gcount = int(m.group("gpus"))

    with metric_csv.open(newline="") as fh:
        r = csv.DictReader(fh)
        rows = list(r)
        cols = r.fieldnames or []

    if not rows or "Time (s)" not in cols:
        return None

    # Drop idle samples
    if "GPU0_DRAM_Active" in cols:
        rows = [
            row for row in rows
            if (f(row.get("GPU0_DRAM_Active")) is not None and f(row.get("GPU0_DRAM_Active")) != 0.0)
        ]
    if not rows:
        return None

    def colvals(rows_in, c):
        out = []
        for row in rows_in:
            v = f(row.get(c))
            if v is not None:
                out.append(v)
        return out

    times = colvals(rows, "Time (s)")
    if not times:
        return None

    # First PROFILE_SECONDS
    t0 = min(times)
    t1 = t0 + PROFILE_SECONDS
    prof = [row for row in rows if (f(row.get("Time (s)")) is not None and f(row.get("Time (s)")) <= t1)]
    if not prof:
        prof = rows

    gpu_ids = sorted({
        int(c.split("_")[0].replace("GPU", ""))
        for c in cols if c.startswith("GPU") and "_" in c
    })
    if len(gpu_ids) >= gcount:
        gpu_ids = gpu_ids[:gcount]
    if not gpu_ids:
        return None

    p_avg, sm_avg, dr_avg = {}, {}, {}
    fp16_avg, fp32_avg, fp64_avg = {}, {}, {}

    for gid in gpu_ids:
        p_avg[gid]    = mean(colvals(prof, f"GPU{gid}_Power (W)"))
        sm_avg[gid]   = mean(colvals(prof, f"GPU{gid}_SM_Clock (MHz)"))
        dr_avg[gid]   = mean(colvals(prof, f"GPU{gid}_DRAM_Active"))
        fp16_avg[gid] = mean(colvals(prof, f"GPU{gid}_FP16_Active"))
        fp32_avg[gid] = mean(colvals(prof, f"GPU{gid}_FP32_Active"))
        fp64_avg[gid] = mean(colvals(prof, f"GPU{gid}_FP64_Active"))

    # Active GPUs
    active = [
        gid for gid in gpu_ids
        if ((not math.isnan(p_avg[gid]) and p_avg[gid] >= ACTIVE_POWER_TH) or
            (not math.isnan(sm_avg[gid]) and sm_avg[gid] >= ACTIVE_SM_TH))
    ]
    if not active:
        active = gpu_ids[:]

    n_active = len(active)

    def safe(v):
        return 0.0 if (v is None or (isinstance(v, float) and math.isnan(v))) else v

    dr_list = [safe(dr_avg[g]) for g in active]
    sm_list = [safe(sm_avg[g]) for g in active]
    fp16_list = [safe(fp16_avg[g]) for g in active]
    fp32_list = [safe(fp32_avg[g]) for g in active]
    fp64_list = [safe(fp64_avg[g]) for g in active]
    fp_list = [fp16_list[i] + fp32_list[i] + fp64_list[i] for i in range(n_active)]

    # Sum across active GPUs
    dram_active_sum = sum(dr_list)
    sm_active_sum = sum(sm_list)
    fp_all_sum = sum(fp_list)

    # Min(active) * num_active
    dram_active_min_scaled = (min(dr_list) * n_active) if n_active else 0.0
    sm_active_min_scaled = (min(sm_list) * n_active) if n_active else 0.0
    fp_all_min_scaled = (min(fp_list) * n_active) if n_active else 0.0

    return {
        "power_cap": cap,
        "gpu_count": gcount,
        "dram_active_sum": dram_active_sum,
        "sm_active_sum": sm_active_sum,
        "fp_all_sum": fp_all_sum,
        "dram_active_min_scaled": dram_active_min_scaled,
        "sm_active_min_scaled": sm_active_min_scaled,
        "fp_all_min_scaled": fp_all_min_scaled,
    }


def run_suite(root, apps, suite_name, kind):
    apps = list_apps(root, apps)
    if not apps:
        print(f"[skip] no apps in {root}")
        return

    for app in apps:
        app_dir = root / app
        if not app_dir.exists():
            continue

        if kind == "runtime":
            target_lookup = read_runtime_lookup_spec(app_dir)
            better = "lower"
        else:
            target_lookup = read_throughput_lookup_ml(app_dir)
            better = "higher"

        by_cap = defaultdict(list)

        for p in sorted(app_dir.iterdir()):
            if not FILE_RE.match(p.name):
                continue

            x = extract_metrics(p)
            if x is None:
                continue

            key = (x["power_cap"], x["gpu_count"])
            target = target_lookup.get(key, None)
            if target is None:
                continue

            x["performance"] = target
            by_cap[x["power_cap"]].append(x)

        if not by_cap:
            continue

        print(f"\n===== {suite_name}/{app} =====")
        print(f"(performance better when {better})")

        for cap in sorted(by_cap.keys()):
            rows = by_cap[cap]
            if kind == "runtime":
                rank_rows = sorted(rows, key=lambda r: r["performance"])  # lower runtime is better
            else:
                rank_rows = sorted(rows, key=lambda r: r["performance"], reverse=True)  # higher throughput is better

            true_rank = [r["gpu_count"] for r in rank_rows]
            print(f"\ncap={cap}\ttrue_rank={true_rank}")
            print("gpu_count\tperformance\tdram_sum\tsm_sum\tfp_sum\tdram_min_scaled\tsm_min_scaled\tfp_min_scaled")
            for r in rank_rows:
                print(
                    f"{r['gpu_count']}\t{fmt2(r['performance'])}\t"
                    f"{fmt2(r['dram_active_sum'])}\t{fmt2(r['sm_active_sum'])}\t{fmt2(r['fp_all_sum'])}\t"
                    f"{fmt2(r['dram_active_min_scaled'])}\t{fmt2(r['sm_active_min_scaled'])}\t{fmt2(r['fp_all_min_scaled'])}"
                )


# Capture output and save txt, while still showing in notebook
buf = StringIO()
with redirect_stdout(buf):
    run_suite(SPEC_ROOT_DIR, SPEC_APPS, "SPEC", "runtime")
    run_suite(ML_ROOT_DIR, ML_APPS, "ML", "throughput")

text = buf.getvalue()
TXT_OUT_PATH.write_text(text)

# print(text, end="")
# print("\n[written]", TXT_OUT_PATH)

42512